# Generating Azure OCR annotations for the FUNSD dataset [prebuilt-layout] 

**Key Requirements:**
1. SDK Version ≥ 1.0.0b4 (Python)
2. API Version ≥ 2024-07-31-preview
3. Model ID must be "prebuilt-layout" for markdown support[4]

**Implementation Notes:**

1. **Markdown Structure:**
- Tables are converted to markdown format with proper alignment[1][6]
- Headers use `#` notation for hierarchy[3]
- Lists maintain their original formatting[3]

2. **API Version Compatibility:**
- Ensure you're using API version `2024-07-31-preview` or newer[6]
- Verify the `ContentFormat` enum is imported from the SDK


3. **Result Handling:**
```python
# To access markdown directly:
markdown_content = result.content

# In your JSON output, look for:
# {
#   "content": "# Markdown Content...",
#   "analyzeResult": { ... }
# }
```



## Import modules and load environment variables

In [4]:
# check Azure sdk  version
import azure.ai.documentintelligence
print(azure.ai.documentintelligence.__version__)


1.0.0b4


In [ ]:
import os
import sys
import json
from dotenv import load_dotenv
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import (
    AnalyzeDocumentRequest,
    AnalyzeResult, 
    DocumentAnalysisFeature,
    ContentFormat # needed for markdown output
)
from azure.core.credentials import AzureKeyCredential

sys.path.append('../src')
from utils import *

# Load the environment variables
load_dotenv()
endpoint = os.getenv('ENDPOINT')
key = os.getenv('COGNITIVE_SERVICE_KEY')

## Parameters

In [10]:
# Define the dataset URL and root directory
DATASET_URL = 'https://www.crc.nd.edu/~pmoreira/FUNSD/dataset'
DATASET_ROOT = '/mnt/slow_data/pmoreira/datasets/FUNSD/'

# Training data directories
TRAIN_DIR = os.path.join(DATASET_ROOT, 'training_data')
TRAIN_ANNOT_DIR = os.path.join(TRAIN_DIR, 'annotations')
TRAIN_IMG_DIR = os.path.join(TRAIN_DIR, 'images')
TRAIN_IMG_URL = f"{DATASET_URL}/training_data/images/"

# Testing data directories
TEST_DIR = os.path.join(DATASET_ROOT, 'testing_data')
TEST_ANNOT_DIR = os.path.join(TEST_DIR, 'annotations')
TEST_IMG_DIR = os.path.join(TEST_DIR, 'images')
TEST_IMG_URL = f"{DATASET_URL}/testing_data/images/"

## Generate OCR annotations for the FUNSD Training dataset

In [ ]:
# Main execution

# Define the dataset directories
dataset_dir = TRAIN_DIR
annot_dir = TRAIN_ANNOT_DIR
img_dir = TRAIN_IMG_DIR
img_url = TRAIN_IMG_URL

# Define Prebuilt model ID
model_id = "prebuilt-layout"

# Create the directory to save the annotations
azure_annot_dir = os.path.join(dataset_dir, f"annotations_azure_model__{model_id.replace('-','_')}_markdown")
os.makedirs(azure_annot_dir, exist_ok=True)

# Initialize the client
client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key),
    api_version="2024-07-31-preview"
)

# Generate the image URLs to analyze
form_urls = generate_image_urls(img_dir, img_url)

# Analyze the documents and save annotations
for form_url in form_urls:
    
    # Analyze the document
    # result, status = analyze_document(form_url, client, model_id)
    poller = client.begin_analyze_document(
    model_id,
    AnalyzeDocumentRequest(url_source=form_url),
    features=[DocumentAnalysisFeature.KEY_VALUE_PAIRS,
            DocumentAnalysisFeature.BARCODES,
            DocumentAnalysisFeature.LANGUAGES],
    string_index_type="utf16CodeUnit",
    output_content_format=ContentFormat.MARKDOWN, # needed for markdown output=, in case only text is needed change to ContentFormat.TEXT
    )
    result: AnalyzeResult= poller.result()

    # Save the result to a JSON file
    annotations_fpath = os.path.join(azure_annot_dir, f"{os.path.basename(form_url).split('.')[0]}.json")
    
    # save_result_to_json(result, annotations_fpath)    
    json_result = json.dumps(result.as_dict(), indent=4)
    with open(annotations_fpath, 'w') as f:
        f.write(json_result)


## Generate OCR annotations for the FUNSD Testing dataset

In [8]:
# Main execution

# Define the dataset directories
dataset_dir = TEST_DIR
annot_dir = TEST_ANNOT_DIR
img_dir = TEST_IMG_DIR
img_url = TEST_IMG_URL

# Define Prebuilt model ID
model_id = "prebuilt-layout"

# Create the directory to save the annotations
azure_annot_dir = os.path.join(dataset_dir, f"annotations_azure_model__{model_id.replace('-','_')}_markdown")
os.makedirs(azure_annot_dir, exist_ok=True)

# Initialize the client
client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key),
    api_version="2024-07-31-preview"
)

# Generate the image URLs to analyze
form_urls = generate_image_urls(img_dir, img_url)

# Analyze the documents and save annotations
for form_url in form_urls:
    
    # Analyze the document
    poller = client.begin_analyze_document(
    model_id,
    AnalyzeDocumentRequest(url_source=form_url),
    features=[DocumentAnalysisFeature.KEY_VALUE_PAIRS,
            DocumentAnalysisFeature.BARCODES,
            DocumentAnalysisFeature.LANGUAGES],
        string_index_type="utf16CodeUnit"
    )
    result: AnalyzeResult= poller.result()

    # Save the result to a JSON file
    annotations_fpath = os.path.join(azure_annot_dir, f"{os.path.basename(form_url).split('.')[0]}.json")

    # save_result_to_json(result, annotations_fpath)    
    json_result = json.dumps(result.as_dict(), indent=4)
    with open(annotations_fpath, 'w') as f:
        f.write(json_result)
